# Занятие 7, пара 1. Видео: кадры, фон, движение


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Картинка получила время

На фотографии мы искали области. В видео появляется новый вопрос: что изменилось между моментами?
Но изменение пикселей ещё не означает движение предмета. Включили свет — изменилась почти вся картинка.
Посылка остановилась — изменение исчезло, хотя посылка никуда не делась.

Разберём это на крупной синтетической сцене конвейера. Сначала число и маленький пример, потом
полный кадр и проверка предположений. Нужны NumPy, Matplotlib, маски и знакомое сравнение изображений.
К концу пары вы сможете отличать межкадровую разность от сравнения с фоном, правильно учитывать время,
обнаруживать ограничение компенсации освещения и задавать устойчивое во времени событие.
Результат — маски с объяснением ошибок и три самостоятельных решения. GPU, камера и интернет не нужны.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Посмотрим на сцену целиком

В `conveyor.npz` лежат сами RGB-кадры, частота и пустой фон. Это массивы, а не видеофайл,
поэтому декодер не нужен. `np.load(..., allow_pickle=False)` открывает именованные массивы.
Размер `frames` — (время, высота, ширина, каналы). Изготовление и условия использования — в
`video/SOURCE.md`. GIF — только предпросмотр: измерять время по скорости его воспроизведения нельзя.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

clip = np.load(DATA_DIR / 'video' / 'conveyor.npz', allow_pickle=False)
frames = clip['frames']
background_rgb = clip['background']
fps = float(clip['fps'])
print('Кадры:', frames.shape, 'Частота:', fps)
display(Image(filename=str(DATA_DIR / 'video' / 'preview.gif')))


Ниже тот же опыт без анимации. Посылка доезжает до остановки, стоит, снова движется,
а затем весь кадр становится светлее. Это разные события. Пока просто назовите их глазами:
не нужно немедленно придумывать «умный алгоритм».


In [ ]:
indices = [0, 12, 20, 26, 31, 47]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for axis, index in zip(axes.ravel(), indices):
    axis.imshow(frames[index])
    axis.set_title('Кадр ' + str(index))
    axis.axis('off')
plt.tight_layout()
plt.show()


### 2. Номер кадра — не время

При постоянных 12 fps соседние кадры разделены 1/12 секунды. Первый имеет время 0, последний —
(N−1)/fps. Длительность показа N кадров по 1/fps равна N/fps: не смешиваем эти две величины.
`np.arange(N)` создаёт номера 0…N−1. `np.diff` вычитает соседние элементы: из [0, 0.2, 0.5]
получатся [0.2, 0.3]. При пропусках кадров используем фактические временные отметки.


In [ ]:
times = np.arange(len(frames)) / fps
print('Время последнего кадра:', times[-1])
print('Длительность показа:', len(frames) / fps)
print('Промежутки времени:', np.diff(np.array([0., 0.2, 0.5])))


### ✏️ Ваш ход — скорость после пропуска кадров

Система сохранила только три наблюдения: времена [0, 0.2, 0.5] секунды,
координаты x=[40,64,100] пикселей. Коллега считает каждый промежуток равным 1/20 секунды.
Напишите `my_velocity(times, positions)`: возвращает скорости между наблюдениями в пикселях/секунду,
с сохранением знака направления. При повторном или убывающем времени выдайте ValueError.
Объясните, почему здесь нельзя измерить метры/секунду и почему нельзя восстановить движение внутри пропуска.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.1', my_velocity)


**Мой вывод:** …


### 3. Что вычитаем и в каком типе

Кадры uint8 хранят 0…255. Разность может быть отрицательной, поэтому **до вычитания** переходим
к float32. Для первого опыта возьмём среднее RGB: это простое учебное сведение к одному каналу,
не точная модель воспринимаемой яркости. `axis=2` сворачивает каналы, сохраняя высоту и ширину.
Модуль разности отвечает на вопрос «насколько изменилось», теряя знак изменения.


In [ ]:
before = np.array([240, 20], dtype=np.uint8)
after = np.array([20, 240], dtype=np.uint8)
signed = after.astype(np.float32) - before.astype(np.float32)
print('Со знаком:', signed)
print('По величине:', np.abs(signed))

gray_frames = []
for frame in frames:
    gray_frames.append(frame.astype(np.float32).mean(axis=2))
gray_frames = np.array(gray_frames)
background = background_rgb.astype(np.float32).mean(axis=2)


Сравним два вопроса на кадре остановки.
**Разность соседей:** что изменилось с предыдущего наблюдения?
**Разность с пустым фоном:** что не похоже на предоставленную пустую сцену?
Порог 25 означает 25 уровней нашей шкалы 0…255, не 25 процентов. Остановившаяся посылка
остаётся на маске второго метода. В первом на движении возникают следы и старого, и нового положения.


In [ ]:
index = 24
neighbor_difference = np.abs(gray_frames[index] - gray_frames[index-1])
background_difference = np.abs(gray_frames[index] - background)
motion_mask = neighbor_difference > 25
presence_mask = background_difference > 25
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frames[index])
axes[0].set_title('Посылка остановилась')
axes[1].imshow(motion_mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Изменение с прошлого кадра')
axes[2].imshow(presence_mask, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Отличие от пустого фона')
for axis in axes:
    axis.axis('off')
plt.show()


### 4. Свет включили. Посылок стало больше?

На кадре 35 прежний фон уже не соответствует освещению. Маска почти сплошная, хотя предмет один.
Просто поднять порог — значит одновременно рисковать потерей слабоконтрастного предмета.
Попробуем конкретную модель: большая часть кадра — прежний фон, к которому добавилась примерно
одинаковая яркость. Тогда медиана **знаковой** разности оценит общий сдвиг света.


In [ ]:
index = 35
delta = gray_frames[index] - background
light_shift = np.median(delta)
fixed_mask = np.abs(delta) > 25
aligned_mask = np.abs(delta - light_shift) > 25
print('Оценённое изменение освещения:', light_shift)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frames[index])
axes[0].set_title('После изменения света')
axes[1].imshow(fixed_mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Неизменный фон')
axes[2].imshow(aligned_mask, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Фон с общей поправкой')
for axis in axes:
    axis.axis('off')
plt.show()


Почему медиана? Если большинство разностей относится к фону, несколько тысяч пикселей посылки
не определяют середину распределения. Но мы не доказали, что фон всегда составляет большинство.
Неравномерная тень, движение камеры и большой передний план тоже нарушают эту модель.
Соберём уже разобранные строки в функцию для следующего опыта; порог по-прежнему в уровнях яркости.


In [ ]:
def aligned_foreground(current, reference, threshold=25):
    difference = current.astype(np.float32) - reference.astype(np.float32)
    shift = np.median(difference)
    return np.abs(difference - shift) > threshold


### ✏️ Ваш ход — сломать разумное предположение

Придумайте крупную сцену 120×180: фон имеет яркость 50,
предмет — 150. Предмет занимает больше половины, но меньше 95% кадра; расположение выберите сами.
Создайте bool-маску `my_truth`, изображение `my_scene` и пустой `my_background`.
Освещение на самом деле не менялось. Примените aligned_foreground и покажите результат рядом с истиной.
Объясните, какую часть сцены функция ошибочно приняла за фон. Предложите изменение условий съёмки
или способ оценки света, а не очередной перебор порога.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.2', my_scene, my_background, my_truth, aligned_foreground)


**Мой вывод:** …


### 5. Событие — это правило над временем

Маска на одном кадре не обязана быть сигналом тревоги. Например, хотим сообщать о присутствии
объекта только после трёх последовательных положительных наблюдений. Однократный блик не проходит.
Счётчик увеличивается на True и сбрасывается на False. Сигнал возникает при переходе счётчика
к трём, а не на каждом следующем кадре. После разрыва может начаться новый эпизод.
На равномерном видео три кадра покрывают два межкадровых промежутка; это не «три секунды».


### ✏️ Ваш ход — один сигнал на эпизод

Напишите `my_alarm_starts(flags, required)`: индексы кадров,
на которых последовательность впервые достигла required значений True подряд.
Для [False,True,False,True,True,True,True,False,True,True,True] и required=3 нужны [5,10].
Коллега возвращает все индексы True, другой проверяет лишь общее число True в ролике. Объясните
ошибку обоих. required — положительное целое; пустой вход должен дать пустой список.
Дополнительно обсудите, как изменить контракт для неравномерных временных отметок.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.3', my_alarm_starts)


**Мой вывод:** …


### 6. Если движется сама камера

Межкадровая разность не различает движение посылки и камеры. Смоделируем сдвиг всего кадра
через `np.roll(..., axis=1)`: он переносит столбцы, а край заворачивает. Чтобы не анализировать
искусственный завёрнутый край, отбросим первые 8 столбцов обоих сравниваемых изображений.
Большой отклик теперь появляется на неподвижных деталях конвейера.


In [ ]:
original = gray_frames[12]
shifted = np.roll(original, 8, axis=1)
camera_mask = np.abs(shifted[:,8:] - original[:,8:]) > 25
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(shifted[:,8:],cmap='gray',vmin=0,vmax=255)
axes[0].set_title('Сдвинутый кадр, край обрезан')
axes[1].imshow(camera_mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Отклик неподвижных деталей')
plt.show()


**Оптический поток** отвечает на другой вопрос: куда сместилась локальная структура изображения?
Это поле двумерных векторов, а не маска «изменилось/не изменилось». Движение камеры тоже входит
в наблюдаемый поток. Чтобы оценить перенос окна, можно искать похожее окно следующего кадра —
знакомая идея сопоставления из блока 05. Методы Lucas–Kanade и Farneback делают это иначе и с
дополнительными предположениями о яркости и согласованном движении соседей.

Ниже резерв: реально оценим горизонтальный сдвиг одного текстурного окна, перебрав кандидатов.
Это ограниченный поиск по одной оси, **не реализация плотного оптического потока**. Вертикальный
сдвиг в этом опыте предполагается нулевым. Вычитаем среднее из обоих окон перед MSE,
чтобы сравнивать рисунок, а не только общую яркость.


In [ ]:
patch = gray_frames[8,192:224,136:168]
patch_centered = patch - patch.mean()
shifts = np.arange(-14,15)
costs = []
for dx in shifts:
    candidate = gray_frames[9,192:224,136+dx:168+dx]
    candidate_centered = candidate - candidate.mean()
    costs.append(np.mean((patch_centered-candidate_centered)**2))
best_dx = int(shifts[np.argmin(costs)])
print('Измеренный сдвиг окна:', best_dx, 'px между кадрами')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(shifts, costs, marker='.')
axes[0].set(xlabel='Сдвиг, px', ylabel='MSE после вычитания среднего')
axes[1].imshow(frames[8])
axes[1].arrow(152,208,best_dx,0,color='red',width=2,head_width=12,length_includes_head=True)
axes[1].set_title('Оценка для одного окна')
plt.show()


### Дополнительно: как это выглядит с OpenCV

Основной путь уже выполнен. `VideoCapture` читает AVI последовательно; `read` возвращает
признак успеха и кадр BGR, `release` закрывает файл. Количество/частота берутся из контейнера,
но могут быть недоступны для конкретного потока. Здесь проверяем чтение локальной учебной копии.
MOG2 поддерживает изменяемую модель фона; параметр history относится к обучению модели,
а не к длительности видео. В примере обучаем её на предоставленном пустом фоне.
При detectShadows=True маска может содержать 127 для тени и 255 для переднего плана:
`mask>0` тогда не равносильно «только передний план». Это другая модель, не наша медианная поправка.


In [ ]:
try:
    import cv2
except (ImportError, OSError):
    cv2 = None

if cv2 is None or not hasattr(cv2,'createBackgroundSubtractorMOG2'):
    print('OpenCV/MOG2 недоступен. Основной опыт выше от этого не зависит.')
else:
    capture = cv2.VideoCapture(str(DATA_DIR / 'video' / 'conveyor.avi'))
    try:
        ok, first_bgr = capture.read()
        if not ok:
            print('Кодек не прочитал AVI; используйте массивы кадров.')
        else:
            print('Частота из контейнера:', capture.get(cv2.CAP_PROP_FPS))
            plt.figure(figsize=(9,5))
            plt.imshow(cv2.cvtColor(first_bgr,cv2.COLOR_BGR2RGB))
            plt.show()
    finally:
        capture.release()
    model = cv2.createBackgroundSubtractorMOG2(history=100,detectShadows=True)
    empty_bgr = cv2.cvtColor(background_rgb,cv2.COLOR_RGB2BGR)
    for repeat in range(20):
        model.apply(empty_bgr)
    mog_mask = model.apply(cv2.cvtColor(frames[12],cv2.COLOR_RGB2BGR),learningRate=0)
    plt.figure(figsize=(9,5))
    plt.imshow(mog_mask==255,cmap='gray',vmin=0,vmax=1)
    plt.title('MOG2: только метка 255')
    plt.show()


**Итог.** Сохраните свою тетрадь, контрпример и объяснение различий между движением,
присутствием и событием. На следующей паре превратим маску в рамку, будем вести объект
между кадрами и считать пересечения линии. Текущее упражнение не потребуется запускать заново.

Официальные примеры по желанию: [модели фона](https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html),
[оптический поток](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html).
Это направления для расширения, не дополнительная обязательная работа.
